# 2016 to 2025 Labor Market: OECD Analysis - Data Prep

In [1]:
import pandas as pd

Common functions

In [2]:
def check_no_duplicates(df, key_cols):
    counts = df[key_cols].value_counts()
    duplicates = counts[counts > 1]
    if duplicates.empty:
        print(f"No duplicates found across {key_cols}")
    else:
        print(f"WARNING: {len(duplicates)} duplicate combinations found:")
        print(duplicates)

In [3]:
def check_no_nulls(df, label=""):
    null_counts = df.isnull().sum()
    has_nulls = null_counts[null_counts > 0]
    prefix = f"[{label}] " if label else ""
    if has_nulls.empty:
        print(f"{prefix}No missing values found.")
    else:
        print(f"{prefix}WARNING: Missing values detected:")
        print(has_nulls.to_string())

In [4]:
def export_by_sex(df, base_path):
    for label, value in [('female', 'Female'), ('male', 'Male'), ('total', 'Total')]:
        subdf = df[df['sex'] == value]
        subdf.to_csv(f'../data/processed/{base_path}_{label}_preprocessed.csv', index=False)

Common dictionaries

In [5]:
EMPLOYMENT_RENAME = {
    'STRUCTURE_NAME': 'structure_name',
    'REF_AREA': 'country_code',
    'Reference area': 'country',
    'Measure': 'measure',
    'Unit of measure': 'unit_measure',
    'Adjustment': 'adjustment',
    'Sex': 'sex',
    'Age': 'age_group',
    'Frequency of observation': 'frequency',
    'TIME_PERIOD': 'time_period',
    'OBS_VALUE': 'employment_value',
    'Observation status': 'observation_status',
}

## OECD Employment Quarterly Data

#### Loading and visualizing table

In [6]:
df_oecd_employ_q = pd.read_csv("../data/raw/oecd_employ_quarter_data.csv")
df_oecd_employ_q.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,MEASURE,Measure,UNIT_MEASURE,Unit of measure,...,OBS_VALUE,Observation value,BASE_PER,Base period,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals
0,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,EMP,Employment,PS,Persons,...,16049.0,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero
1,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,EMP,Employment,PS,Persons,...,16090.0,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero
2,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,EMP,Employment,PS,Persons,...,16083.0,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero
3,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,EMP,Employment,PS,Persons,...,16093.0,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero
4,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,EMP,Employment,PS,Persons,...,16140.0,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero


In [7]:
print(df_oecd_employ_q.columns)
print(df_oecd_employ_q.shape)

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA',
       'Reference area', 'MEASURE', 'Measure', 'UNIT_MEASURE',
       'Unit of measure', 'TRANSFORMATION', 'Transformation', 'ADJUSTMENT',
       'Adjustment', 'SEX', 'Sex', 'AGE', 'Age', 'ACTIVITY',
       'Economic activity', 'FREQ', 'Frequency of observation', 'TIME_PERIOD',
       'Time period', 'OBS_VALUE', 'Observation value', 'BASE_PER',
       'Base period', 'OBS_STATUS', 'Observation status', 'UNIT_MULT',
       'Unit multiplier', 'DECIMALS', 'Decimals'],
      dtype='str')
(18204, 34)


In [8]:
df_oecd_employ_q.describe()

,Time period,OBS_VALUE,Observation value,BASE_PER,Base period,UNIT_MULT,DECIMALS
count,0.0,18204.000000,0.0,0.0,0.0,18204.0,18204.0
mean,NaN,5223.303299,NaN,NaN,NaN,3.0,0.0
std,NaN,12087.300033,NaN,NaN,NaN,0.0,0.0
min,NaN,7.000000,NaN,NaN,NaN,3.0,0.0
25%,NaN,327.113275,NaN,NaN,NaN,3.0,0.0
50%,NaN,1366.596500,NaN,NaN,NaN,3.0,0.0
75%,NaN,4400.988250,NaN,NaN,NaN,3.0,0.0
max,NaN,152098.700000,NaN,NaN,NaN,3.0,0.0


In [9]:
df_oecd_employ_q['Reference area'].unique()

<StringArray>
[ 'United Kingdom',          'Canada',         'Czechia',         'Finland',
          'Greece',           'Italy',        'Portugal',         'Austria',
         'Denmark',           'Spain',         'Belgium',         'Hungary',
 'Slovak Republic',        'Slovenia',         'Ireland',          'Norway',
         'Estonia',     'Netherlands',          'Poland',          'Sweden',
          'Latvia',          'France',         'Iceland',       'Lithuania',
          'Mexico',         'Türkiye',      'Luxembourg',     'Switzerland',
      'Costa Rica',           'Japan',           'Korea',          'Israel',
       'Australia',     'New Zealand',         'Germany',   'United States',
        'Colombia',           'Chile']
Length: 38, dtype: str

### Preprocessing

#### Checking first 6 columns: STRUCTURE, STRUCTURE_ID, STRUCTURE_NAME, ACTION, REF_AREA, Reference area

In [10]:
df_oecd_employ_q[['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA', 'Reference area']]

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area
0,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom
1,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom
2,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom
3,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom
4,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom
...,...,...,...,...,...,...
18199,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,CHL,Chile
18200,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,CHL,Chile
18201,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,CHL,Chile
18202,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,CHL,Chile


Technical metadata fields associated with the SDMX standard (STRUCTURE, STRUCTURE_ID, and ACTION), except for STRUCTURE_NAME, are removed during preprocessing because they contain administrative information only and do not provide analytical value for the study.

In [11]:
cols_to_drop_q = [
    'STRUCTURE', 'STRUCTURE_ID', 'ACTION'
]

#### Checking next 6 columns: MEASURE, Measure, UNIT_MEASURE, Unit of measure, TRANSFORMATION, Transformation

In [12]:
df_oecd_employ_q[['MEASURE', 'Measure', 'UNIT_MEASURE', 'Unit of measure',
         'TRANSFORMATION', 'Transformation']]

,MEASURE,Measure,UNIT_MEASURE,Unit of measure,TRANSFORMATION,Transformation
0,EMP,Employment,PS,Persons,_Z,Not applicable
1,EMP,Employment,PS,Persons,_Z,Not applicable
2,EMP,Employment,PS,Persons,_Z,Not applicable
3,EMP,Employment,PS,Persons,_Z,Not applicable
4,EMP,Employment,PS,Persons,_Z,Not applicable
...,...,...,...,...,...,...
18199,EMP,Employment,PS,Persons,_Z,Not applicable
18200,EMP,Employment,PS,Persons,_Z,Not applicable
18201,EMP,Employment,PS,Persons,_Z,Not applicable
18202,EMP,Employment,PS,Persons,_Z,Not applicable


Duplicated SDMX code columns (MEASURE and UNIT_MEASURE) are removed while retaining their corresponding descriptive label columns (Measure and Unit of measure). This improves dataset readability without loss of analytical information.

In [13]:
df_oecd_employ_q = df_oecd_employ_q.drop(columns=['MEASURE', 'UNIT_MEASURE'])

The TRANSFORMATION and Transformation columns are also removed because they contain only the constant value "Not Applicable", indicating that no additional mathematical transformation had been applied to the employment indicators.

In [14]:
df_oecd_employ_q['Transformation'].unique(), df_oecd_employ_q['TRANSFORMATION'].unique()

(<StringArray>
 ['Not applicable']
 Length: 1, dtype: str,
 <StringArray>
 ['_Z']
 Length: 1, dtype: str)

In [15]:
cols_to_drop_q.extend(['Transformation', 'TRANSFORMATION'])

#### Checking next 6 columns: ADJUSTMENT, Adjustment, SEX, Sex, AGE, Age

In [16]:
df_oecd_employ_q[['ADJUSTMENT', 'Adjustment', 'SEX', 'Sex', 'AGE', 'Age']]

,ADJUSTMENT,Adjustment,SEX,Sex,AGE,Age
0,Y,Calendar and seasonally adjusted,M,Male,Y15T64,From 15 to 64 years
1,Y,Calendar and seasonally adjusted,M,Male,Y15T64,From 15 to 64 years
2,Y,Calendar and seasonally adjusted,M,Male,Y15T64,From 15 to 64 years
3,Y,Calendar and seasonally adjusted,M,Male,Y15T64,From 15 to 64 years
4,Y,Calendar and seasonally adjusted,M,Male,Y15T64,From 15 to 64 years
...,...,...,...,...,...,...
18199,Y,Calendar and seasonally adjusted,_T,Total,Y15T64,From 15 to 64 years
18200,Y,Calendar and seasonally adjusted,_T,Total,Y15T64,From 15 to 64 years
18201,Y,Calendar and seasonally adjusted,_T,Total,Y15T64,From 15 to 64 years
18202,Y,Calendar and seasonally adjusted,_T,Total,Y15T64,From 15 to 64 years


In [17]:
df_oecd_employ_q['AGE'].unique()

<StringArray>
['Y15T64', 'Y55T64', 'Y25T54', 'Y15T24']
Length: 4, dtype: str

The strategy of removing duplicated technical metadata, leaving only human-redable information, is applied from here on.

In [18]:
cols_to_drop_q.extend(['ADJUSTMENT', 'SEX', 'AGE'])

#### Checking next 6 columns: ACTIVITY, Economic activity, FREQ, Frequency of observation, TIME_PERIOD, Time period

In [19]:
df_oecd_employ_q[['ACTIVITY', 'Economic activity', 'FREQ', 'Frequency of observation',
                  'TIME_PERIOD', 'Time period']]

,ACTIVITY,Economic activity,FREQ,Frequency of observation,TIME_PERIOD,Time period
0,_Z,Not applicable,Q,Quarterly,2016-Q1,NaN
1,_Z,Not applicable,Q,Quarterly,2016-Q2,NaN
2,_Z,Not applicable,Q,Quarterly,2016-Q3,NaN
3,_Z,Not applicable,Q,Quarterly,2016-Q4,NaN
4,_Z,Not applicable,Q,Quarterly,2017-Q1,NaN
...,...,...,...,...,...,...
18199,_Z,Not applicable,Q,Quarterly,2024-Q4,NaN
18200,_Z,Not applicable,Q,Quarterly,2025-Q1,NaN
18201,_Z,Not applicable,Q,Quarterly,2025-Q2,NaN
18202,_Z,Not applicable,Q,Quarterly,2025-Q3,NaN


The ACTIVITY and Economic activity columns are removed because they contain only the constant value "_Z" or "Not Applicable", indicating that the dataset did not include an industry or sector-level breakdown of employment.

The FREQ column is also removed because it contains only the constant value "Q", and its information is also brought by column Frequency of observation.

The Time period column only contains a constant value of NaN and can be considered redundant since "TIME_PERIOD" has the information.

In [20]:
df_oecd_employ_q['Economic activity'].unique()

<StringArray>
['Not applicable']
Length: 1, dtype: str

In [21]:
df_oecd_employ_q['Time period'].unique()

array([nan])

In [22]:
df_oecd_employ_q["FREQ"].unique()

<StringArray>
['Q']
Length: 1, dtype: str

In [23]:
cols_to_drop_q.extend(['ACTIVITY', 'Economic activity', 'FREQ', 'Time period'])

Splitting time period into year and quarter

In [24]:
df_oecd_employ_q["TIME_PERIOD"]

0        2016-Q1
1        2016-Q2
2        2016-Q3
3        2016-Q4
4        2017-Q1
          ...   
18199    2024-Q4
18200    2025-Q1
18201    2025-Q2
18202    2025-Q3
18203    2025-Q4
Name: TIME_PERIOD, Length: 18204, dtype: str

In [25]:
df_oecd_employ_q[['year', 'quarter']] = df_oecd_employ_q['TIME_PERIOD'].str.split('-', expand=True)
df_oecd_employ_q['year'] = df_oecd_employ_q['year'].astype(int)
df_oecd_employ_q.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,Measure,Unit of measure,TRANSFORMATION,Transformation,...,BASE_PER,Base period,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals,year,quarter
0,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,Employment,Persons,_Z,Not applicable,...,NaN,NaN,A,Normal value,3,Thousands,0,Zero,2016,Q1
1,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,Employment,Persons,_Z,Not applicable,...,NaN,NaN,A,Normal value,3,Thousands,0,Zero,2016,Q2
2,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,Employment,Persons,_Z,Not applicable,...,NaN,NaN,A,Normal value,3,Thousands,0,Zero,2016,Q3
3,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,Employment,Persons,_Z,Not applicable,...,NaN,NaN,A,Normal value,3,Thousands,0,Zero,2016,Q4
4,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,GBR,United Kingdom,Employment,Persons,_Z,Not applicable,...,NaN,NaN,A,Normal value,3,Thousands,0,Zero,2017,Q1


#### Checking next 6 columns: OBS_VALUE, Observation value, BASE_PER, Base period, OBS_STATUS, Observation status

In [26]:
df_oecd_employ_q[['OBS_VALUE', 'Observation value', 'BASE_PER', 'Base period',
         'OBS_STATUS', 'Observation status']]

,OBS_VALUE,Observation value,BASE_PER,Base period,OBS_STATUS,Observation status
0,16049.000,NaN,NaN,NaN,A,Normal value
1,16090.000,NaN,NaN,NaN,A,Normal value
2,16083.000,NaN,NaN,NaN,A,Normal value
3,16093.000,NaN,NaN,NaN,A,Normal value
4,16140.000,NaN,NaN,NaN,A,Normal value
...,...,...,...,...,...,...
18199,8677.885,NaN,NaN,NaN,E,Estimated value
18200,8765.798,NaN,NaN,NaN,E,Estimated value
18201,8777.450,NaN,NaN,NaN,E,Estimated value
18202,8831.285,NaN,NaN,NaN,E,Estimated value


In [27]:
df_oecd_employ_q['Observation value'].unique()

array([nan])

In [28]:
df_oecd_employ_q['BASE_PER'].unique()

array([nan])

In [29]:
df_oecd_employ_q['Base period'].unique()

array([nan])

In [30]:
df_oecd_employ_q['Observation status'].unique()

<StringArray>
['Normal value', 'Estimated value']
Length: 2, dtype: str

The dataset includes both officially reported (‘Normal value’) and statistically estimated observations (‘Estimated value’). Estimated observations are retained in the analysis to preserve time-series continuity, while an indicator flag is created to identify potentially revised or modeled values in downstream visualizations and analysis.

In [31]:
df_oecd_employ_q['is_estimated'] = (
    df_oecd_employ_q['Observation status'] == 'Estimated value'
)

Columns related to SDMX base-period metadata (BASE_PER and Base period) are removed because they contain no values and are not applicable to the employment-level indicators analyzed in this study. The Observation value column is also removed due to the absence of populated observations.

In [32]:
cols_to_drop_q.extend(['Observation value', 'BASE_PER', 'Base period', 'OBS_STATUS'])

#### Checking last 4 columns: UNIT_MULT, Unit multiplier, DECIMALS, Decimals

In [33]:
df_oecd_employ_q[['UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals']]

,UNIT_MULT,Unit multiplier,DECIMALS,Decimals
0,3,Thousands,0,Zero
1,3,Thousands,0,Zero
2,3,Thousands,0,Zero
3,3,Thousands,0,Zero
4,3,Thousands,0,Zero
...,...,...,...,...
18199,3,Thousands,0,Zero
18200,3,Thousands,0,Zero
18201,3,Thousands,0,Zero
18202,3,Thousands,0,Zero


In [34]:
df_oecd_employ_q['UNIT_MULT'].unique()

array([3])

In [35]:
df_oecd_employ_q['Unit multiplier'].unique()

<StringArray>
['Thousands']
Length: 1, dtype: str

In [36]:
df_oecd_employ_q['DECIMALS'].unique()

array([0])

In [37]:
df_oecd_employ_q['Decimals'].unique()

<StringArray>
['Zero']
Length: 1, dtype: str

Metadata columns related to unit scaling (UNIT_MULT, Unit multiplier) and display precision (DECIMALS, Decimals) are removed because they contain constant values across all observations and do not provide additional analytical information. Employment values remain expressed in thousands of persons, as defined by the OECD metadata.

In [38]:
cols_to_drop_q.extend(['UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals'])

#### Dropping selected columns

In [39]:
df_oecd_employ_q = df_oecd_employ_q.drop(columns=cols_to_drop_q)

#### Final column check

In [40]:
columns_to_check = [
    'REF_AREA', 'Reference area', 'Measure', 'Unit of measure',
    'Adjustment', 'Sex', 'Age', 'Frequency of observation', 'TIME_PERIOD',
    'Observation status', 'year', 'quarter', 'is_estimated'
]

for col in columns_to_check:
    print(f"Unique values for '{col}':")
    print(df_oecd_employ_q[col].unique())
    print("\n")

Unique values for 'REF_AREA':
<StringArray>
['GBR', 'CAN', 'CZE', 'FIN', 'GRC', 'ITA', 'PRT', 'AUT', 'DNK', 'ESP', 'BEL',
 'HUN', 'SVK', 'SVN', 'IRL', 'NOR', 'EST', 'NLD', 'POL', 'SWE', 'LVA', 'FRA',
 'ISL', 'LTU', 'MEX', 'TUR', 'LUX', 'CHE', 'CRI', 'JPN', 'KOR', 'ISR', 'AUS',
 'NZL', 'DEU', 'USA', 'COL', 'CHL']
Length: 38, dtype: str


Unique values for 'Reference area':
<StringArray>
[ 'United Kingdom',          'Canada',         'Czechia',         'Finland',
          'Greece',           'Italy',        'Portugal',         'Austria',
         'Denmark',           'Spain',         'Belgium',         'Hungary',
 'Slovak Republic',        'Slovenia',         'Ireland',          'Norway',
         'Estonia',     'Netherlands',          'Poland',          'Sweden',
          'Latvia',          'France',         'Iceland',       'Lithuania',
          'Mexico',         'Türkiye',      'Luxembourg',     'Switzerland',
      'Costa Rica',           'Japan',           'Korea',          'Isra

#### Checking rows for duplicates

In [41]:
check_no_duplicates(df_oecd_employ_q, ['Reference area', 'Sex', 'Age', 'TIME_PERIOD'])

No duplicates found across ['Reference area', 'Sex', 'Age', 'TIME_PERIOD']


### Final check

In [42]:
df_oecd_employ_q

,STRUCTURE_NAME,REF_AREA,Reference area,Measure,Unit of measure,Adjustment,Sex,Age,Frequency of observation,TIME_PERIOD,OBS_VALUE,Observation status,year,quarter,is_estimated
0,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2016-Q1,16049.000,Normal value,2016,Q1,False
1,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2016-Q2,16090.000,Normal value,2016,Q2,False
2,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2016-Q3,16083.000,Normal value,2016,Q3,False
3,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2016-Q4,16093.000,Normal value,2016,Q4,False
4,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2017-Q1,16140.000,Normal value,2017,Q1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18199,Employed population by age groups,CHL,Chile,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Quarterly,2024-Q4,8677.885,Estimated value,2024,Q4,True
18200,Employed population by age groups,CHL,Chile,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Quarterly,2025-Q1,8765.798,Estimated value,2025,Q1,True
18201,Employed population by age groups,CHL,Chile,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Quarterly,2025-Q2,8777.450,Estimated value,2025,Q2,True
18202,Employed population by age groups,CHL,Chile,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Quarterly,2025-Q3,8831.285,Estimated value,2025,Q3,True


In [43]:
df_oecd_employ_q.columns

Index(['STRUCTURE_NAME', 'REF_AREA', 'Reference area', 'Measure',
       'Unit of measure', 'Adjustment', 'Sex', 'Age',
       'Frequency of observation', 'TIME_PERIOD', 'OBS_VALUE',
       'Observation status', 'year', 'quarter', 'is_estimated'],
      dtype='str')

#### Reordering columns

In [44]:
df_oecd_employ_q = df_oecd_employ_q[
  [
    'STRUCTURE_NAME',
    'REF_AREA', 'Reference area',
    'Measure', 'Unit of measure',
    'Adjustment', 'Sex',
    'Age', 'Frequency of observation',
    'TIME_PERIOD', 'year', 'quarter',
    'OBS_VALUE', 'Observation status', 'is_estimated'
  ]
]

#### Checking for missing values

In [45]:
check_no_nulls(df_oecd_employ_q, label="employ_q")

[employ_q] No missing values found.


#### Standardizing column names for better readability

In [46]:
df_oecd_employ_q = df_oecd_employ_q.rename(columns=EMPLOYMENT_RENAME)

In [47]:
df_oecd_employ_q.columns

Index(['structure_name', 'country_code', 'country', 'measure', 'unit_measure',
       'adjustment', 'sex', 'age_group', 'frequency', 'time_period', 'year',
       'quarter', 'employment_value', 'observation_status', 'is_estimated'],
      dtype='str')

## Final dataset

In [48]:
df_oecd_employ_q

,structure_name,country_code,country,measure,unit_measure,adjustment,sex,age_group,frequency,time_period,year,quarter,employment_value,observation_status,is_estimated
0,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2016-Q1,2016,Q1,16049.000,Normal value,False
1,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2016-Q2,2016,Q2,16090.000,Normal value,False
2,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2016-Q3,2016,Q3,16083.000,Normal value,False
3,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2016-Q4,2016,Q4,16093.000,Normal value,False
4,Employed population by age groups,GBR,United Kingdom,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 64 years,Quarterly,2017-Q1,2017,Q1,16140.000,Normal value,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18199,Employed population by age groups,CHL,Chile,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Quarterly,2024-Q4,2024,Q4,8677.885,Estimated value,True
18200,Employed population by age groups,CHL,Chile,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Quarterly,2025-Q1,2025,Q1,8765.798,Estimated value,True
18201,Employed population by age groups,CHL,Chile,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Quarterly,2025-Q2,2025,Q2,8777.450,Estimated value,True
18202,Employed population by age groups,CHL,Chile,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Quarterly,2025-Q3,2025,Q3,8831.285,Estimated value,True


In [49]:
df_oecd_employ_q.info()

<class 'pandas.DataFrame'>
RangeIndex: 18204 entries, 0 to 18203
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   structure_name      18204 non-null  str    
 1   country_code        18204 non-null  str    
 2   country             18204 non-null  str    
 3   measure             18204 non-null  str    
 4   unit_measure        18204 non-null  str    
 5   adjustment          18204 non-null  str    
 6   sex                 18204 non-null  str    
 7   age_group           18204 non-null  str    
 8   frequency           18204 non-null  str    
 9   time_period         18204 non-null  str    
 10  year                18204 non-null  int64  
 11  quarter             18204 non-null  str    
 12  employment_value    18204 non-null  float64
 13  observation_status  18204 non-null  str    
 14  is_estimated        18204 non-null  bool   
dtypes: bool(1), float64(1), int64(1), str(12)
memory usage: 2.0 MB


In [50]:
df_oecd_employ_q.describe()

,year,employment_value
count,18204.000000,18204.000000
mean,2020.491101,5223.303299
std,2.868212,12087.300033
min,2016.000000,7.000000
25%,2018.000000,327.113275
50%,2020.000000,1366.596500
75%,2023.000000,4400.988250
max,2025.000000,152098.700000


#### Exporting dataset in csv format

In [51]:
df_oecd_employ_q.to_csv('../data/processed/oecd_employ_quarter_preprocessed.csv', index=False)

#### Splitting dataset by sex and total

In [52]:
export_by_sex(df_oecd_employ_q, './oecd_employ_quarter')

# OECD Employment Yearly Data

#### Loading and visualizing table

In [53]:
df_oecd_employ_y = pd.read_csv("../data/raw/oecd_employ_year_data.csv")
df_oecd_employ_y.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,MEASURE,Measure,UNIT_MEASURE,Unit of measure,...,OBS_VALUE,Observation value,BASE_PER,Base period,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals
0,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,USA,United States,EMP,Employment,PS,Persons,...,142519.7,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero
1,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,USA,United States,EMP,Employment,PS,Persons,...,144103.3,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero
2,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,USA,United States,EMP,Employment,PS,Persons,...,146055.5,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero
3,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,USA,United States,EMP,Employment,PS,Persons,...,147191.3,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero
4,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),Employed population by age groups,I,USA,United States,EMP,Employment,PS,Persons,...,137976.8,NaN,NaN,NaN,A,Normal value,3,Thousands,0,Zero


In [54]:
df_oecd_employ_y.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA',
       'Reference area', 'MEASURE', 'Measure', 'UNIT_MEASURE',
       'Unit of measure', 'TRANSFORMATION', 'Transformation', 'ADJUSTMENT',
       'Adjustment', 'SEX', 'Sex', 'AGE', 'Age', 'ACTIVITY',
       'Economic activity', 'FREQ', 'Frequency of observation', 'TIME_PERIOD',
       'Time period', 'OBS_VALUE', 'Observation value', 'BASE_PER',
       'Base period', 'OBS_STATUS', 'Observation status', 'UNIT_MULT',
       'Unit multiplier', 'DECIMALS', 'Decimals'],
      dtype='str')

In [55]:
df_oecd_employ_y.describe()

,TIME_PERIOD,Time period,OBS_VALUE,Observation value,BASE_PER,Base period,UNIT_MULT,DECIMALS
count,4464.000000,0.0,4464.000000,0.0,0.0,0.0,4464.0,4464.0
mean,2020.477151,NaN,5143.977649,NaN,NaN,NaN,3.0,0.0
std,2.865014,NaN,11832.225051,NaN,NaN,NaN,0.0,0.0
min,2016.000000,NaN,8.325000,NaN,NaN,NaN,3.0,0.0
25%,2018.000000,NaN,321.875000,NaN,NaN,NaN,3.0,0.0
50%,2020.000000,NaN,1347.262500,NaN,NaN,NaN,3.0,0.0
75%,2023.000000,NaN,4376.231250,NaN,NaN,NaN,3.0,0.0
max,2025.000000,NaN,150157.300000,NaN,NaN,NaN,3.0,0.0


## Preprocessing

```df_oecd_employ_y``` and ```df_oecd_employ_q``` share many columns and are treated similarly. ```df_oecd_employ_y``` is checked for the dropped columns from ```df_oecd_employ_q```.

#### Checking first 7 columns: STRUCTURE, STRUCTURE_ID, ACTION, MEASURE, UNIT_MEASURE, Transformation, TRANSFORMATION

In [56]:
df_oecd_employ_y[['STRUCTURE', 'STRUCTURE_ID', 'ACTION', 'MEASURE', 'UNIT_MEASURE', 'TRANSFORMATION', 'Transformation']]

,STRUCTURE,STRUCTURE_ID,ACTION,MEASURE,UNIT_MEASURE,TRANSFORMATION,Transformation
0,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),I,EMP,PS,_Z,Not applicable
1,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),I,EMP,PS,_Z,Not applicable
2,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),I,EMP,PS,_Z,Not applicable
3,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),I,EMP,PS,_Z,Not applicable
4,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),I,EMP,PS,_Z,Not applicable
...,...,...,...,...,...,...,...
4459,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),I,EMP,PS,_Z,Not applicable
4460,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),I,EMP,PS,_Z,Not applicable
4461,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),I,EMP,PS,_Z,Not applicable
4462,DATAFLOW,OECD.SDD.TPS:DSD_LFS@DF_IALFS_EMP_Q(1.0),I,EMP,PS,_Z,Not applicable


In [57]:
for col in ['ACTION', 'MEASURE', 'UNIT_MEASURE', 'TRANSFORMATION', 'Transformation']:
    print(f"Unique values for '{col}':")
    print(df_oecd_employ_y[col].unique())
    print("\n")

Unique values for 'ACTION':
<StringArray>
['I']
Length: 1, dtype: str


Unique values for 'MEASURE':
<StringArray>
['EMP']
Length: 1, dtype: str


Unique values for 'UNIT_MEASURE':
<StringArray>
['PS']
Length: 1, dtype: str


Unique values for 'TRANSFORMATION':
<StringArray>
['_Z']
Length: 1, dtype: str


Unique values for 'Transformation':
<StringArray>
['Not applicable']
Length: 1, dtype: str




In [58]:
cols_to_drop_y = [
    'STRUCTURE', 'STRUCTURE_ID', 'ACTION',
    'MEASURE', 'UNIT_MEASURE', 'TRANSFORMATION', 'Transformation'
]

#### Checking next 5 columns: ADJUSTMENT, SEX, AGE, ACTIVITY, Economic activity

In [59]:
df_oecd_employ_y[['ADJUSTMENT', 'SEX', 'AGE', 'ACTIVITY', 'Economic activity']]

,ADJUSTMENT,SEX,AGE,ACTIVITY,Economic activity
0,Y,_T,Y15T64,_Z,Not applicable
1,Y,_T,Y15T64,_Z,Not applicable
2,Y,_T,Y15T64,_Z,Not applicable
3,Y,_T,Y15T64,_Z,Not applicable
4,Y,_T,Y15T64,_Z,Not applicable
...,...,...,...,...,...
4459,Y,M,Y15T24,_Z,Not applicable
4460,Y,_T,Y25T54,_Z,Not applicable
4461,Y,_T,Y55T64,_Z,Not applicable
4462,Y,M,Y55T64,_Z,Not applicable


In [60]:
for col in ['ADJUSTMENT', 'SEX', 'AGE', 'ACTIVITY', 'Economic activity']:
    print(f"Unique values for '{col}':")
    print(df_oecd_employ_y[col].unique())
    print("\n")

Unique values for 'ADJUSTMENT':
<StringArray>
['Y']
Length: 1, dtype: str


Unique values for 'SEX':
<StringArray>
['_T', 'M', 'F']
Length: 3, dtype: str


Unique values for 'AGE':
<StringArray>
['Y15T64', 'Y55T64', 'Y25T54', 'Y15T24']
Length: 4, dtype: str


Unique values for 'ACTIVITY':
<StringArray>
['_Z']
Length: 1, dtype: str


Unique values for 'Economic activity':
<StringArray>
['Not applicable']
Length: 1, dtype: str




Creating upper and lower bounds + age midpoint

In [61]:
cols_to_drop_y.extend(['ADJUSTMENT', 'SEX', 'AGE', 'ACTIVITY', 'Economic activity'])

#### Checking next 7 columns: FREQ, TIME_PERIOD, Time period, Observation value, BASE_PER, Base period, OBS_STATUS

In [62]:
df_oecd_employ_y[['FREQ', 'TIME_PERIOD', 'Time period', 'Observation value', 'BASE_PER', 'Base period', 'OBS_STATUS']]

,FREQ,TIME_PERIOD,Time period,Observation value,BASE_PER,Base period,OBS_STATUS
0,A,2016,NaN,NaN,NaN,NaN,A
1,A,2017,NaN,NaN,NaN,NaN,A
2,A,2018,NaN,NaN,NaN,NaN,A
3,A,2019,NaN,NaN,NaN,NaN,A
4,A,2020,NaN,NaN,NaN,NaN,A
...,...,...,...,...,...,...,...
4459,A,2025,NaN,NaN,NaN,NaN,A
4460,A,2025,NaN,NaN,NaN,NaN,A
4461,A,2025,NaN,NaN,NaN,NaN,A
4462,A,2025,NaN,NaN,NaN,NaN,A


In [63]:
for col in ['FREQ', 'TIME_PERIOD', 'Time period', 'Observation value', 'BASE_PER', 'Base period']:
    print(f"Unique values for '{col}':")
    print(df_oecd_employ_y[col].unique())
    print("\n")

Unique values for 'FREQ':
<StringArray>
['A']
Length: 1, dtype: str


Unique values for 'TIME_PERIOD':
[2016 2017 2018 2019 2020 2021 2022 2023 2024 2025]


Unique values for 'Time period':
[nan]


Unique values for 'Observation value':
[nan]


Unique values for 'BASE_PER':
[nan]


Unique values for 'Base period':
[nan]




Taking a closer look at OBS_STATUS

In [64]:
print(df_oecd_employ_y['OBS_STATUS'].unique())

<StringArray>
['A', 'B', 'U']
Length: 3, dtype: str


```df_oecd_employ_y``` has different Observation status than ```df_oecd_employ_q```

In [65]:
df_oecd_employ_y[['OBS_STATUS', 'Observation status']].groupby('OBS_STATUS').value_counts()

OBS_STATUS  Observation status
A           Normal value          4332
B           Time series break      120
U           Low reliability         12
Name: count, dtype: int64

A time series break means that the methodology, definition, survey design, classification, or data source changed at some point.
Low reliability means OECD considers the estimate statistically weak or uncertain. It's usually caused by small sample size, survey volatility, high standard error, estimation issues and/or low response rates. The number is still usable, but confidence is lower.
These observations represent a small portion of the data and will be kept on the analysis under the flag ```is_estimated```.

In [66]:
df_oecd_employ_y['is_estimated'] = df_oecd_employ_y['Observation status'].isin([
    'Break in time series', 'Low reliability'
])

In [67]:
cols_to_drop_y.extend(['FREQ', 'Time period', 'Observation value',
                     'BASE_PER', 'Base period', 'OBS_STATUS'])

#### Checking last 4 columns: UNIT_MULT, Unit multiplier, DECIMALS, Decimals

In [68]:
df_oecd_employ_y[['UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals']]

,UNIT_MULT,Unit multiplier,DECIMALS,Decimals
0,3,Thousands,0,Zero
1,3,Thousands,0,Zero
2,3,Thousands,0,Zero
3,3,Thousands,0,Zero
4,3,Thousands,0,Zero
...,...,...,...,...
4459,3,Thousands,0,Zero
4460,3,Thousands,0,Zero
4461,3,Thousands,0,Zero
4462,3,Thousands,0,Zero


In [69]:
for col in ['UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals']:
    print(f"Unique values for '{col}':")
    print(df_oecd_employ_y[col].unique())
    print("\n")

Unique values for 'UNIT_MULT':
[3]


Unique values for 'Unit multiplier':
<StringArray>
['Thousands']
Length: 1, dtype: str


Unique values for 'DECIMALS':
[0]


Unique values for 'Decimals':
<StringArray>
['Zero']
Length: 1, dtype: str




In [70]:
cols_to_drop_y.extend(['UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals'])

#### Dropping selected columns

In [71]:
df_oecd_employ_y = df_oecd_employ_y.drop(columns=cols_to_drop_y)

#### Checking rows for duplicates

In [72]:
check_no_duplicates(df_oecd_employ_y, ['Reference area', 'Sex', 'Age', 'TIME_PERIOD'])

No duplicates found across ['Reference area', 'Sex', 'Age', 'TIME_PERIOD']


### Final check

In [73]:
df_oecd_employ_y

,STRUCTURE_NAME,REF_AREA,Reference area,Measure,Unit of measure,Adjustment,Sex,Age,Frequency of observation,TIME_PERIOD,OBS_VALUE,Observation status,is_estimated
0,Employed population by age groups,USA,United States,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Annual,2016,142519.700,Normal value,False
1,Employed population by age groups,USA,United States,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Annual,2017,144103.300,Normal value,False
2,Employed population by age groups,USA,United States,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Annual,2018,146055.500,Normal value,False
3,Employed population by age groups,USA,United States,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Annual,2019,147191.300,Normal value,False
4,Employed population by age groups,USA,United States,Employment,Persons,Calendar and seasonally adjusted,Total,From 15 to 64 years,Annual,2020,137976.800,Normal value,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4459,Employed population by age groups,NZL,New Zealand,Employment,Persons,Calendar and seasonally adjusted,Male,From 15 to 24 years,Annual,2025,189.575,Normal value,False
4460,Employed population by age groups,NZL,New Zealand,Employment,Persons,Calendar and seasonally adjusted,Total,From 25 to 54 years,Annual,2025,1808.200,Normal value,False
4461,Employed population by age groups,NZL,New Zealand,Employment,Persons,Calendar and seasonally adjusted,Total,From 55 to 64 years,Annual,2025,479.175,Normal value,False
4462,Employed population by age groups,NZL,New Zealand,Employment,Persons,Calendar and seasonally adjusted,Male,From 55 to 64 years,Annual,2025,244.575,Normal value,False


#### Reordering columns

In [74]:
df_oecd_employ_y = df_oecd_employ_y[
  [
    'STRUCTURE_NAME',
    'REF_AREA', 'Reference area',
    'Measure', 'Unit of measure',
    'Adjustment', 'Sex',
    'Age', 'Frequency of observation', 'TIME_PERIOD',
    'OBS_VALUE', 'Observation status', 'is_estimated'
  ]
]

#### Checking for missing values

In [75]:
check_no_nulls(df_oecd_employ_y, label="employ_y")

[employ_y] No missing values found.


#### Standardizing column names for better readability

In [76]:
df_oecd_employ_y = df_oecd_employ_y.rename(columns=EMPLOYMENT_RENAME)

### Final dataset

In [77]:
df_oecd_employ_y.info()

<class 'pandas.DataFrame'>
RangeIndex: 4464 entries, 0 to 4463
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   structure_name      4464 non-null   str    
 1   country_code        4464 non-null   str    
 2   country             4464 non-null   str    
 3   measure             4464 non-null   str    
 4   unit_measure        4464 non-null   str    
 5   adjustment          4464 non-null   str    
 6   sex                 4464 non-null   str    
 7   age_group           4464 non-null   str    
 8   frequency           4464 non-null   str    
 9   time_period         4464 non-null   int64  
 10  employment_value    4464 non-null   float64
 11  observation_status  4464 non-null   str    
 12  is_estimated        4464 non-null   bool   
dtypes: bool(1), float64(1), int64(1), str(10)
memory usage: 423.0 KB


In [78]:
df_oecd_employ_y.describe()

,time_period,employment_value
count,4464.000000,4464.000000
mean,2020.477151,5143.977649
std,2.865014,11832.225051
min,2016.000000,8.325000
25%,2018.000000,321.875000
50%,2020.000000,1347.262500
75%,2023.000000,4376.231250
max,2025.000000,150157.300000


### Exporting dataset in csv format

In [79]:
df_oecd_employ_y.to_csv('../data/processed/oecd_employ_year_preprocessed.csv', index=False)

In [80]:
df_oecd_employ_y.columns

Index(['structure_name', 'country_code', 'country', 'measure', 'unit_measure',
       'adjustment', 'sex', 'age_group', 'frequency', 'time_period',
       'employment_value', 'observation_status', 'is_estimated'],
      dtype='str')

#### Splitting dataset by sex and total

In [81]:
export_by_sex(df_oecd_employ_y, './oecd_employ_year')

## OECD Population Data

#### Loading and visualizing table

In [82]:
df_oecd_pop = pd.read_csv("../data/raw/oecd_population_data.csv")
df_oecd_pop.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,MEASURE,Measure,UNIT_MEASURE,Unit of measure,...,TIME_PERIOD,Time period,OBS_VALUE,Observation value,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals
0,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Historical population data,I,EST,Estonia,POP,Population,PS,Persons,...,2016,NaN,422397.0,NaN,A,Normal value,0,Units,0,Zero
1,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Historical population data,I,EST,Estonia,POP,Population,PS,Persons,...,2017,NaN,421669.0,NaN,A,Normal value,0,Units,0,Zero
2,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Historical population data,I,EST,Estonia,POP,Population,PS,Persons,...,2018,NaN,422821.0,NaN,A,Normal value,0,Units,0,Zero
3,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Historical population data,I,EST,Estonia,POP,Population,PS,Persons,...,2019,NaN,424071.0,NaN,A,Normal value,0,Units,0,Zero
4,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Historical population data,I,EST,Estonia,POP,Population,PS,Persons,...,2020,NaN,424375.0,NaN,A,Normal value,0,Units,0,Zero


In [83]:
df_oecd_pop.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA',
       'Reference area', 'MEASURE', 'Measure', 'UNIT_MEASURE',
       'Unit of measure', 'SEX', 'Sex', 'AGE', 'Age', 'TIME_HORIZ',
       'Time horizon', 'TIME_PERIOD', 'Time period', 'OBS_VALUE',
       'Observation value', 'OBS_STATUS', 'Observation status', 'UNIT_MULT',
       'Unit multiplier', 'DECIMALS', 'Decimals'],
      dtype='str')

In [84]:
df_oecd_pop.describe()

,TIME_PERIOD,Time period,OBS_VALUE,Observation value,UNIT_MULT,DECIMALS
count,1041.000000,0.0,1.041000e+03,0.0,1041.0,1041.0
mean,2020.072046,NaN,1.572785e+07,NaN,0.0,0.0
std,2.632927,NaN,2.745896e+07,NaN,0.0,0.0
min,2016.000000,NaN,1.069370e+05,NaN,0.0,0.0
25%,2018.000000,NaN,1.835785e+06,NaN,0.0,0.0
50%,2020.000000,NaN,5.518394e+06,NaN,0.0,0.0
75%,2022.000000,NaN,1.907544e+07,NaN,0.0,0.0
max,2025.000000,NaN,2.165217e+08,NaN,0.0,0.0


### Preprocessing

A similar column preprocessing to employment data is applied to this dataset. Human-readable metadata columns are kept while coded ones are removed. Constant values of multiplers, decimals, etc, if any, are also removed.

In [85]:
df_oecd_pop.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA',
       'Reference area', 'MEASURE', 'Measure', 'UNIT_MEASURE',
       'Unit of measure', 'SEX', 'Sex', 'AGE', 'Age', 'TIME_HORIZ',
       'Time horizon', 'TIME_PERIOD', 'Time period', 'OBS_VALUE',
       'Observation value', 'OBS_STATUS', 'Observation status', 'UNIT_MULT',
       'Unit multiplier', 'DECIMALS', 'Decimals'],
      dtype='str')

In [86]:
columns = [
    'MEASURE', 'UNIT_MEASURE', 'SEX', 'AGE', 'Age', 'TIME_HORIZ',
    'Time horizon', 'Time period', 'OBS_STATUS', 'Observation status',
    'UNIT_MULT', 'DECIMALS'
]

for col in columns:
    print(f"\nColumn: {col}")
    print(df_oecd_pop[col].unique())


Column: MEASURE
<StringArray>
['POP']
Length: 1, dtype: str

Column: UNIT_MEASURE
<StringArray>
['PS']
Length: 1, dtype: str

Column: SEX
<StringArray>
['M', '_T', 'F']
Length: 3, dtype: str

Column: AGE
<StringArray>
['Y15T64']
Length: 1, dtype: str

Column: Age
<StringArray>
['From 15 to 64 years']
Length: 1, dtype: str

Column: TIME_HORIZ
<StringArray>
['H']
Length: 1, dtype: str

Column: Time horizon
<StringArray>
['Historical']
Length: 1, dtype: str

Column: Time period
[nan]

Column: OBS_STATUS
<StringArray>
['A']
Length: 1, dtype: str

Column: Observation status
<StringArray>
['Normal value']
Length: 1, dtype: str

Column: UNIT_MULT
[0]

Column: DECIMALS
[0]


Creating a flag is_estimated for consistency, even with a Normal constant value

In [87]:
df_oecd_pop['is_estimated'] = df_oecd_pop['Observation status'] != 'Normal value'

#### Dropping selected columns

In [88]:
df_oecd_pop = df_oecd_pop.drop(columns=['STRUCTURE', 'STRUCTURE_ID', 'ACTION',
                                        'MEASURE', 'UNIT_MEASURE',
                                        'SEX', 'TIME_HORIZ', 'AGE',
                                        'Time period', 'Time horizon',
                                        'Observation value', 'OBS_STATUS', 'UNIT_MULT',
                                        'Unit multiplier', 'DECIMALS', 'Decimals'])

### Checking rows for duplicates

In [89]:
check_no_duplicates(df_oecd_pop, ['Reference area', 'Sex', 'Age', 'TIME_PERIOD'])

No duplicates found across ['Reference area', 'Sex', 'Age', 'TIME_PERIOD']


#### Converting OBS_VALUE column to integer

In [90]:
df_oecd_pop['OBS_VALUE'] = (
    df_oecd_pop['OBS_VALUE']
    .round()
    .astype('Int64')
)

#### Converting population values to thousands in order to match employment data

In [91]:
df_oecd_pop['OBS_VALUE'] = df_oecd_pop['OBS_VALUE'] / 1000

#### Reordering columns

In [92]:
df_oecd_pop = df_oecd_pop[
  [
    'STRUCTURE_NAME',
    'REF_AREA', 'Reference area',
    'Measure', 'Unit of measure',
    'Sex', 'Age',
    'TIME_PERIOD',
    'OBS_VALUE', 'Observation status', 'is_estimated'
  ]
]

#### Checking for missing values

In [93]:
check_no_nulls(df_oecd_pop, label="population")

[population] No missing values found.


#### Standardizing column names

In [94]:
df_oecd_pop.columns

Index(['STRUCTURE_NAME', 'REF_AREA', 'Reference area', 'Measure',
       'Unit of measure', 'Sex', 'Age', 'TIME_PERIOD', 'OBS_VALUE',
       'Observation status', 'is_estimated'],
      dtype='str')

In [95]:
df_oecd_pop = df_oecd_pop.rename(columns={
    'STRUCTURE_NAME': 'structure_name',
    'REF_AREA': 'country_code',
    'Reference area': 'country',
    'Measure': 'measure',
    'Unit of measure': 'unit_of_measure',
    'TIME_PERIOD': 'year',
    'Sex': 'sex',
    'Age': 'age',
    'OBS_VALUE': 'population_value',
    'Observation status': 'observation_status'
})

### Final View

In [96]:
df_oecd_pop

,structure_name,country_code,country,measure,unit_of_measure,sex,age,year,population_value,observation_status,is_estimated
0,Historical population data,EST,Estonia,Population,Persons,Male,From 15 to 64 years,2016,422.397,Normal value,False
1,Historical population data,EST,Estonia,Population,Persons,Male,From 15 to 64 years,2017,421.669,Normal value,False
2,Historical population data,EST,Estonia,Population,Persons,Male,From 15 to 64 years,2018,422.821,Normal value,False
3,Historical population data,EST,Estonia,Population,Persons,Male,From 15 to 64 years,2019,424.071,Normal value,False
4,Historical population data,EST,Estonia,Population,Persons,Male,From 15 to 64 years,2020,424.375,Normal value,False
...,...,...,...,...,...,...,...,...,...,...,...
1036,Historical population data,DEU,Germany,Population,Persons,Female,From 15 to 64 years,2021,26247.994,Normal value,False
1037,Historical population data,DEU,Germany,Population,Persons,Total,From 15 to 64 years,2021,53299.642,Normal value,False
1038,Historical population data,DEU,Germany,Population,Persons,Male,From 15 to 64 years,2023,26893.862,Normal value,False
1039,Historical population data,DEU,Germany,Population,Persons,Female,From 15 to 64 years,2023,26165.654,Normal value,False


In [97]:
df_oecd_pop.info()

<class 'pandas.DataFrame'>
RangeIndex: 1041 entries, 0 to 1040
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   structure_name      1041 non-null   str    
 1   country_code        1041 non-null   str    
 2   country             1041 non-null   str    
 3   measure             1041 non-null   str    
 4   unit_of_measure     1041 non-null   str    
 5   sex                 1041 non-null   str    
 6   age                 1041 non-null   str    
 7   year                1041 non-null   int64  
 8   population_value    1041 non-null   Float64
 9   observation_status  1041 non-null   str    
 10  is_estimated        1041 non-null   bool   
dtypes: Float64(1), bool(1), int64(1), str(8)
memory usage: 83.5 KB


In [98]:
df_oecd_pop.describe()

,year,population_value
count,1041.000000,1041.0
mean,2020.072046,15727.851683
std,2.632927,27458.963305
min,2016.000000,106.937
25%,2018.000000,1835.785
50%,2020.000000,5518.394
75%,2022.000000,19075.437
max,2025.000000,216521.68


### Exporting dataset to csv format

In [99]:
df_oecd_pop.to_csv('../data/processed/oecd_population_preprocessed.csv', index=False)

#### Splitting by sex and total

In [100]:
export_by_sex(df_oecd_pop, './oecd_population')